In [1]:
import os
os.chdir("../")

In [30]:
%pwd

'c:\\Users\\narra\\OneDrive\\Desktop\\cibil_score_prediction'

In [4]:
!pip install dagshub mlflow

  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached yarl-1.20.1-cp313-cp313-win_amd64.whl.metadata (76 kB)
  Using cached propcache-0.3.2-cp313-cp313-win_amd64.whl.metadata (12 kB)
Using cached tenacity-9.1.2-py3-none-any.whl (28 kB)
   ---------------------------------------- 0.0/14.0 MB ? eta -:--:--
    --------------------------------------- 0.3/14.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/14.0 MB 2.0 MB/s eta 0:00:07
   -- ------------------------------------- 0.8/14.0 MB 1.3 MB/s eta 0:00:10
   -- ------------------------------------- 0.8/14.0 MB 1.3 MB/s eta 0:00:10
   -- ------------------------------------- 1.0/14.0 MB 1.1 MB/s eta 0:00:12
   --- ------------------------------------ 1.3/14.0 MB 1.0 MB/s eta 0:00:13
   ---- ----------------------------------- 1.6/14.0 MB 1.1 MB/s eta 0:00:12
   ----- ---------------------------------- 1.8/14.0 MB 1.1 MB/s eta 0:00:12
   ----- ---------------------------------- 2.1/14.0 MB 1.1

In [8]:
import dagshub
dagshub.init(repo_owner='narraranjith22', repo_name='cibil_score_prediction', mlflow=True)


Initialized MLflow to track repo "narraranjith22/cibil_score_prediction"

Repository narraranjith22/cibil_score_prediction initialized!

In [31]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [32]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [33]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.RandomForest
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri="https://dagshub.com/narraranjith22/cibil_score_prediction.mlflow",
            
        )

        return model_evaluation_config


In [34]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib


In [35]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        accuracy = accuracy_score(actual, pred)
        cm = confusion_matrix(actual, pred)
        cr = classification_report(actual, pred)
        return accuracy, cm, cr

    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]  # Ensure 1D array for sklearn metrics

        # Set MLflow tracking URI if provided
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)

            accuracy, cm, cr = self.eval_metrics(test_y, predicted_qualities)

            # Saving metrics as local
            scores = {
                "accuracy": accuracy,
                "confusion_matrix": cm.tolist(),
                "classification_report": cr
            }
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("accuracy", accuracy)
            # Log confusion matrix and classification report as artifacts or text, not as metrics
            mlflow.log_text(str(cm), "confusion_matrix.txt")
            mlflow.log_text(cr, "classification_report.txt")

            # Model registry does not work with file store
            #mlflow.sklearn.log_model(model, "model")

In [36]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2025-09-02 01:31:00,555: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-09-02 01:31:00,557: INFO: common: yaml file: params.yaml loaded successfully]
[2025-09-02 01:31:00,559: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-09-02 01:31:00,560: INFO: common: created directory at: artifacts]
[2025-09-02 01:31:00,561: INFO: common: created directory at: artifacts/model_evaluation]
[2025-09-02 01:31:01,698: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]
🏃 View run painted-finch-696 at: https://dagshub.com/narraranjith22/cibil_score_prediction.mlflow/#/experiments/0/runs/7dab4767058a49de8e125d1f66e36026
🧪 View experiment at: https://dagshub.com/narraranjith22/cibil_score_prediction.mlflow/#/experiments/0
